# 2. YAML and CLI

The Python API is for exploration. Production runs the same rules from a YAML file: `veridelta run` in CI, exit 0 or 1, `--json` on stdout, artifacts on disk.

This notebook writes two parquet files, a config, and exercises that path. The Python models are [1. Core Concepts](01_core_concepts.ipynb). CLI flags are in the [Configuration Guide](../configuration.md#command-line).

## 1. Source and target files

Three shared accounts plus one target-only row. Status codes are abbreviated; one balance drifted by 99 cents.

In [ ]:
import polars as pl

pl.DataFrame(
    {
        "user_id": [1, 2, 3],
        "status": ["Active", "Pending", "Closed"],
        "balance": ["$100.50", "$50.00", "$0.00"],
    }
).write_parquet("source.parquet")

pl.DataFrame(
    {
        "user_id": [1, 2, 3, 4],
        "status": ["ACT", "PND", "CLS", "ACT"],
        "balance": [100.50, 50.99, 0.00, 12.00],
    }
).write_parquet("target.parquet")
print("wrote source.parquet and target.parquet")

# Output:
# wrote source.parquet and target.parquet

## 2. Declare the policy

`output_path` is the artifact directory. Empty discrepancy frames are not written; a clean run leaves it empty.

In [ ]:
%%writefile veridelta.yaml
source:
  path: "source.parquet"
  format: "parquet"
target:
  path: "target.parquet"
  format: "parquet"
primary_keys: ["user_id"]
output_path: "./diff_results"
output_format: "parquet"
rules:
  - column_names: ["status"]
    value_map:
      Active: ACT
      Pending: PND
      Closed: CLS
  - column_names: ["balance"]
    regex_replace:
      "\\$": ""
    cast_to: Float64

# Output:
# Writing veridelta.yaml


## 3. Run the CLI

Exit `0` means the comparison fell within `threshold`. Exit `1` means drift, or the run could not complete. `--json` prints `DiffSummary` on stdout; `--quiet` keeps progress off stderr so `veridelta run --json --quiet | jq` is parseable.

In [ ]:
!veridelta run -c veridelta.yaml --quiet

# Output:
#
# Veridelta Execution Summary
# ===========================
# Status:        FAILED
# Match Rate:    33.33%
# Source Rows:   3
# Target Rows:   4
# Volume Shift:  +1 rows
#
# Row-Level Discrepancies:
# ---------------------------
# Added:         1
# Removed:       0
# Changed:       1
# Total Issues:  2
#
# Top Column-Level Drifts:
# ---------------------------
# - balance: 1 mismatches

In [ ]:
!veridelta run -c veridelta.yaml --json --quiet; echo "exit=$?"

# Output:
# {
#   "total_rows_source": 3,
#   "total_rows_target": 4,
#   "added_count": 1,
#   "removed_count": 0,
#   "changed_count": 1,
#   "column_mismatches": {
#     "balance": 1
#   },
#   "is_match": false,
#   "total_mismatches": 2,
#   "mismatch_ratio": 0.6666666666666666,
#   "match_rate_percentage": 33.33,
#   "is_perfect_match": false,
#   "volume_shift": 1,
#   "report_summary": "Veridelta Execution Summary\n===========================\nStatus:        FAILED\nMatch Rate:    33.33%\nSource Rows:   3\nTarget Rows:   4\nVolume Shift:  +1 rows\n\nRow-Level Discrepancies:\n---------------------------\nAdded:         1\nRemoved:       0\nChanged:       1\nTotal Issues:  2\n\nTop Column-Level Drifts:\n---------------------------\n- balance: 1 mismatches\n"
# }
# exit=1

## 4. The same YAML from Python

`load_config` returns the parsed models. `DiffEngine.run_from_configs` is the orchestrator entry point — no `LazyFrame` construction in the caller.

In [ ]:
from veridelta import DiffEngine, load_config

diff_cfg, src_cfg, tgt_cfg = load_config("veridelta.yaml")
summary = DiffEngine.run_from_configs(diff_cfg, src_cfg, tgt_cfg).summary
print(f"is_match={summary.is_match}  changed={summary.changed_count}  added={summary.added_count}")

# Output:
# is_match=False  changed=1  added=1

## 5. Environment override

Keep one reviewed policy file and let each environment say where the data lives. Any string inside `source` or `target` can read an environment variable: `${NAME}`, or `${NAME:-default}` to fall back when the variable is unset or empty. A warehouse `password` stays out of the file the same way. Rules and root settings are read verbatim; see [Environment variables](../configuration.md#environment-variables).

From Python, `model_copy(update={"path": ...})` overrides a loaded config directly.

In [ ]:
%%writefile veridelta.env.yaml
source:
  path: "${VERIDELTA_DATA_ROOT:-.}/source.parquet"
  format: "parquet"
target:
  path: "${VERIDELTA_DATA_ROOT:-.}/target.parquet"
  format: "parquet"
primary_keys: ["user_id"]
rules:
  - column_names: ["status"]
    value_map:
      Active: ACT
      Pending: PND
      Closed: CLS
  - column_names: ["balance"]
    regex_replace:
      "\\$": ""
    cast_to: Float64

# Output:
# Writing veridelta.env.yaml


In [ ]:
import os
import shutil
from pathlib import Path

os.environ.pop("VERIDELTA_DATA_ROOT", None)
_, src_cfg, _ = load_config("veridelta.env.yaml")
print(f"unset: {src_cfg.path}")

Path("data").mkdir(exist_ok=True)
for name in ("source.parquet", "target.parquet"):
    shutil.copy(name, Path("data") / name)
os.environ["VERIDELTA_DATA_ROOT"] = "data"
diff_cfg, src_cfg, tgt_cfg = load_config("veridelta.env.yaml")
summary = DiffEngine.run_from_configs(diff_cfg, src_cfg, tgt_cfg).summary
print(f"set:   {src_cfg.path}")
print(f"is_match={summary.is_match}")

# Output:
# unset: ./source.parquet
# set:   data/source.parquet
# is_match=False

## 6. Artifacts

Non-empty `added`, `removed`, and `changed` frames are written under `output_path`.

In [ ]:
from pathlib import Path

print(sorted(path.name for path in Path("diff_results").iterdir()))

# Output:
# ['added_rows.parquet', 'changed_rows.parquet']

Advanced drift on real data is [3. Advanced Rules](03_advanced_rules.ipynb). The HTML hand-off is [4. HTML Reports](04_html_reports.ipynb).

In [ ]:
import shutil
from pathlib import Path

for name in ("source.parquet", "target.parquet", "veridelta.yaml", "veridelta.env.yaml"):
    Path(name).unlink(missing_ok=True)
shutil.rmtree("diff_results", ignore_errors=True)
shutil.rmtree("data", ignore_errors=True)
print("cleaned")

# Output:
# cleaned